# Co-authorship: Collaboration Metrics and Network Structure

This notebook examines how biomedical research is authored: how team sizes have changed, how collaboration has grown, and what the structure of co-authorship looks like among prolific authors.

It is built in two parts for a sound reason. A single co-authorship graph over all 3 million articles is not tractable: author lists range up to roughly 2,929 names, and one large-consortium paper alone generates millions of co-author pairs (k authors make k-choose-2 edges), which would dominate the graph and exhaust memory. So Part A computes collaboration metrics across the full corpus (robust, no graph), and Part B builds an actual co-authorship network on a deliberately scoped subset (post-2014, normal-sized teams, prolific authors only).

Two limits set by the data, carried from the EDA. Author names are not disambiguated: "J Smith" at one institution and "J Smith" at another are the same node, so this is a name-level network, not a person-level one. And affiliation coverage is only reliable after 2014, so the network is restricted to that range.

Runs on the published metadata (author_names is included; no abstracts needed).

## Setup

In [ ]:
import os, glob, collections, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx   

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING: pick ONE option (same pattern as 03_eda / 04)
# =====================================================================

# ---- OPTION A: LOCAL (active) ----
ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, "data", "2_clean")

# ---- OPTION B: KAGGLE (uncomment on Kaggle) ----
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input; attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])
# =====================================================================

df = pd.read_parquet(DATA_DIR, columns=["uid", "year", "author_names", "n_authors"])
df = df[df["year"] <= 2025].copy()
print(f"loaded {len(df):,} records")
assert len(df) == df["uid"].nunique(), "duplicate PMIDs; dedup did not run"
df[["year", "n_authors"]].head(3)

# Part A: Collaboration metrics (full corpus)

These use every article and need no graph, so they are robust and fast. They describe how team sizes and collaboration patterns have changed across the full 1994-2025 range.

## 1. Team size over time

Mean and median authors per paper per year, plus the share of solo and large-team papers. The mean is sensitive to the large-consortia tail, so the median is the more stable measure of the typical paper.

In [ ]:
by_year = df.groupby("year")["n_authors"]
metrics = pd.DataFrame({
    "mean":   by_year.mean(),
    "median": by_year.median(),
})
metrics["pct_solo"]  = df.assign(s=df["n_authors"] == 1).groupby("year")["s"].mean() * 100
metrics["pct_large"] = df.assign(l=df["n_authors"] >= 10).groupby("year")["l"].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.lineplot(data=metrics[["mean", "median"]], dashes=False, markers=True, ax=axes[0])
axes[0].set_title("Authors per paper (mean vs median)"); axes[0].set_xlabel("year"); axes[0].set_ylabel("authors")
sns.lineplot(data=metrics[["pct_solo", "pct_large"]], dashes=False, markers=True, ax=axes[1])
axes[1].set_title("Solo (1 author) vs large (10+ author) papers"); axes[1].set_xlabel("year"); axes[1].set_ylabel("% of papers")
plt.tight_layout(); plt.show()
print(metrics.loc[metrics.index.isin([1995, 2005, 2015, 2025])].round(1).to_string())

**What this shows:** the typical paper (median) gains authors only slowly, but the mean rises faster and more erratically because the large-consortia tail grows. Solo authorship declines steadily while the share of 10+ author papers climbs, confirming the shift toward team science seen in the EDA. The gap between mean and median is itself a signal: it widens as large collaborations become more common.

## 2. Team-size composition

The full distribution of team sizes by band, as a share of each year's papers. This shows not just that teams grew, but which size classes gained and which shrank.

In [ ]:
band = pd.cut(df["n_authors"], [0, 1, 5, 10, 20, 10**9],
              labels=["solo", "2-5", "6-10", "11-20", "21+"])
comp = df.assign(band=band).groupby(["year", "band"], observed=True).size().unstack(fill_value=0)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100

comp_pct.plot.area(figsize=(13, 5), alpha=0.8,
                   color=["#d9534f", "#1d6fb8", "#2a9d5c", "#8a5fb0", "#e0853f"])
plt.title("Team-size composition by year (%)"); plt.xlabel("year"); plt.ylabel("% of papers")
plt.legend(title="authors", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()
print(comp_pct.loc[comp_pct.index.isin([1995, 2005, 2015, 2025])].round(1).to_string())

**What this shows:** the bands redistribute over time. Solo and small (2-5) papers lose share while mid and large bands grow, so the "average" rise is a genuine shift in how research is organised, not just a few mega-papers pulling the mean. The composition view is more honest than the mean alone because it shows where the change actually happens.

# Part B: Co-authorship network (scoped subset)

A real co-authorship graph, built only after excluding the cases that would make it intractable or misleading: large-consortium papers (which create giant cliques and explode the edge count), pre-2014 records (unreliable affiliations), and one-off authors (who add noise without structure). The result is a network of how prolific authors collaborate, not a complete map of all authorship.

## 3. Why the cap is necessary

In [ ]:
# Build freq (author frequencies) and pair (co-author counts).
# Required for all subsequent network and diagnostic cells.
import collections
import itertools
from tqdm.auto import tqdm
import numpy as np

MAX_AUTHORS = 50      # same cap as the diagnostic; exclude giant consortia (k-choose-2 explodes)

# Same scope as the network: post-2014, normal-sized teams
sub = df[(df["year"] >= 2014) & (df["n_authors"] <= MAX_AUTHORS)]
print(f"papers in scope (post-2014, <= {MAX_AUTHORS} authors): {len(sub):,} "
      f"({len(df) - len(sub):,} excluded)")

freq = collections.Counter()
pair = collections.Counter()

for names in tqdm(sub["author_names"], total=len(sub), desc="counting co-authorship"):
    # author_names may load as a list or a numpy array; handle both (and a stray string)
    if isinstance(names, str):
        authors = [a.strip() for a in names.split(";") if a.strip()]
    elif isinstance(names, (list, np.ndarray)):
        authors = list(names)
    else:
        continue
    u = sorted(set(authors))
    freq.update(u)
    pair.update(itertools.combinations(u, 2))

print(f"distinct author names: {len(freq):,}")
print(f"distinct co-author pairs: {len(pair):,}")

In [ ]:
import math
print("co-author pairs generated by one paper, by author count:")
for n in [5, 10, 20, 50, 100, 500, 2929]:
    print(f"  {n:>5} authors -> {math.comb(n, 2):>12,} pairs")

print("\nhow many papers exceed each author count:")
for n in [10, 20, 50, 100, 200, 500]:
    cnt = (df["n_authors"] > n).sum()
    print(f"  > {n:>3} authors: {cnt:>9,} papers ({cnt/len(df)*100:.2f}%)")

**What this shows:** A paper with k authors generates **k-choose-2 = k(k-1)/2** co-author pairs:

| Authors | Pairs | Paper count > this |
|---------|-------|-------------------|
| 10 | 45 | 333,614 (10.9%) |
| 20 | 190 | 56,466 (1.8%) |
| 50 | 1,225 | 4,419 (0.14%) |
| 100 | 4,950 | 870 (0.03%) |
| 500 | 124,750 | 13 (0.00%) |
| 2,929 | **4,288,056** | 1 paper |

**The critical insight:** Excluding papers above 50 authors removes only 0.14% of papers but prevents a single consortia paper from generating more co-author pairs than thousands of normal papers combined. This is not arbitrary trimming—it's essential for tractability.

## 4. Build the scoped network

The graph keeps post-2014 papers with at most MAX_AUTHORS authors, then restricts to authors with at least MIN_PAPERS papers (the recurring, prolific names) and edges of at least MIN_COLLAB joint papers (removing one-off collaborations). The thresholds are explicit and can be tuned.

In [ ]:
MIN_PAPERS = 100      # 2,477 authors clear this (from the diagnostic)
MIN_COLLAB = 20       # strong repeated collaborations only
MAX_NODES  = 500      # safety cap: do not attempt to draw more than this many nodes

core = set(a for a, c in freq.items() if c >= MIN_PAPERS)
G = nx.Graph()
for (a, b), c in pair.items():
    if a in core and b in core and c >= MIN_COLLAB:
        G.add_edge(a, b, weight=c)
if G.number_of_nodes():
    comps = sorted(nx.connected_components(G), key=len, reverse=True)
    G = G.subgraph(comps[0]).copy()

print(f"scoped network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# SAFETY: refuse to draw an unreadable graph
if G.number_of_nodes() > MAX_NODES:
    print(f"\nWARNING: {G.number_of_nodes()} nodes is too many to draw readably.")
    print(f"Raise MIN_PAPERS / MIN_COLLAB until nodes <= {MAX_NODES}, then re-run.")

In [ ]:
try:
    import networkx as nx
except ImportError:
    print("networkx not installed; run `pip install networkx` to build the graph.")
    nx = None

if nx is not None:
    core = set(a for a, c in freq.items() if c >= MIN_PAPERS)
    G = nx.Graph()
    for (a, b), c in pair.items():
        if a in core and b in core and c >= MIN_COLLAB:
            G.add_edge(a, b, weight=c)
    if G.number_of_nodes():
        comps = sorted(nx.connected_components(G), key=len, reverse=True)
        G = G.subgraph(comps[0]).copy()      # largest connected component for a clean layout

    print(f"scoped network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    if G.number_of_nodes():
        cent = nx.degree_centrality(G)
        print("\nmost connected authors (most distinct collaborators):")
        for a, v in sorted(cent.items(), key=lambda x: -x[1])[:10]:
            print(f"  {a:<30} {v:.3f}")

In [ ]:
if nx is not None and G.number_of_nodes():
    cent = nx.degree_centrality(G)

    print("MOST connected authors (widest variety of collaborators):")
    for a, v in sorted(cent.items(), key=lambda x: -x[1])[:10]:
        print(f"  {a:<30} {v:.3f}  ({G.degree(a)} collaborators)")

    print("\nLEAST connected authors in the network (tight-team specialists):")
    for a, v in sorted(cent.items(), key=lambda x: x[1])[:10]:
        print(f"  {a:<30} {v:.3f}  ({G.degree(a)} collaborators)")

    print("\nTIGHTEST collaborations (most co-published author pairs):")
    top_edges = sorted(G.edges(data=True), key=lambda e: -e[2]["weight"])[:10]
    for u, v, d in top_edges:
        print(f"  {u}  +  {v}  :  {d['weight']} joint papers")

**What this shows:** The most connected and tightest-paired authors are recognisable real research leaders, not name-collision artifacts:

- **Heart failure trials:** McMurray, Solomon, Zannad, Anker, Butler — large-scale clinical trial consortia
- **Alzheimer's biomarkers:** Zetterberg, Blennow, Jack, Petersen — international biomarker collaborations
- **The Blennow-Zetterberg pair (562 joint papers)** is a well-documented real collaboration, not an artifact

The high prolific-author threshold (>=100 papers) filters out common-name collisions. While name-level ambiguity remains a structural limit, at this threshold the top entries correspond to identifiable researchers.

In [ ]:
if nx is not None and G.number_of_nodes() and G.number_of_nodes() <= MAX_NODES:
    pos = nx.spring_layout(G, k=0.5, seed=42, weight="weight")
    sizes = [200 + 4000 * cent[n] for n in G.nodes()]
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    wmax = max(weights) if weights else 1
    widths = [0.3 + 3.0 * (w / wmax) for w in weights]

    plt.figure(figsize=(14, 11))
    nx.draw_networkx_edges(G, pos, width=widths, alpha=0.2, edge_color="#888")
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color="#1d6fb8", alpha=0.85)
    big = dict(sorted(cent.items(), key=lambda x: -x[1])[:20])    # label only top 20
    nx.draw_networkx_labels(G.subgraph(big), pos, font_size=7)
    plt.title("Co-authorship network among prolific authors (post-2014, capped)")
    plt.axis("off"); plt.tight_layout(); plt.show()

elif nx is not None and G.number_of_nodes() > MAX_NODES:
    print(f"skipping plot: {G.number_of_nodes()} nodes exceeds MAX_NODES={MAX_NODES}. Tighten thresholds.")

## 5. Collaboration communities

Community detection finds clusters of authors who collaborate more within the group than outside it: research groups, labs, or tight collaborator circles. The number and size of communities describe how fragmented or interconnected the prolific-author landscape is.

In [ ]:
from tqdm.auto import tqdm

core = set(a for a, c in freq.items() if c >= MIN_PAPERS)
G = nx.Graph()
for (a, b), c in tqdm(pair.items(), total=len(pair), desc="building edges"):
    if a in core and b in core and c >= MIN_COLLAB:
        G.add_edge(a, b, weight=c)

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

# G is already built (cells 16/22). Detect communities on it.
communities = list(greedy_modularity_communities(G, weight="weight"))
community_id = {node: idx for idx, comm in enumerate(communities) for node in comm}

sizes = sorted((len(c) for c in communities), reverse=True)
print(f"found {len(communities)} collaboration communities")
print(f"largest community sizes: {sizes[:10]}")
print(f"median community size: {int(np.median(sizes))}")

print("\ntop 5 authors per community (by degree centrality):")
cent = nx.degree_centrality(G)
for idx, comm in enumerate(communities):
    top_nodes = sorted(comm, key=lambda x: cent.get(x, 0), reverse=True)[:5]
    print(f"  community {idx+1}: {', '.join(top_nodes)}")

# Plot the network coloured by community (only if small enough to read)
if G.number_of_nodes() <= MAX_NODES:
    pos = nx.spring_layout(G, k=0.5, seed=42, weight="weight")
    cmap = plt.colormaps["tab20"]
    ncomm = max(len(communities), 1)
    colors = [cmap(community_id[n] % 20) for n in G.nodes()]

    weights = [G[u][v]["weight"] for u, v in G.edges()]
    wmax = max(weights) if weights else 1
    widths = [0.3 + 3.0 * (w / wmax) for w in weights]

    plt.figure(figsize=(14, 11))
    nx.draw_networkx_edges(G, pos, width=widths, alpha=0.15, edge_color="#888")
    nx.draw_networkx_nodes(G, pos, node_size=250, node_color=colors, alpha=0.9)
    plt.title("Co-authorship network coloured by collaboration community")
    plt.axis("off"); plt.tight_layout(); plt.show()
else:
    print(f"\nskipping plot: {G.number_of_nodes()} nodes exceeds MAX_NODES={MAX_NODES}.")

**What this shows:** The prolific author network resolves into collaboration communities, each a cluster of authors who work together more than with outsiders. Many small to medium communities indicate a landscape of distinct research groups rather than one interconnected mass. The communities identified align with recognisable research specialities, from Alzheimer's biomarker consortia to cardio oncology and immunology networks, confirming that these name level clusters genuinely reflect real scientific teams. As with the network itself, these groupings remain name based and should be interpreted with the caveats below in mind.

## 6. Summary and Conclusions

### What This Analysis Reveals

**Part A – Full Corpus Collaboration Trends**
- The typical paper has grown from 3 to 7 authors (1995–2025), while the mean has risen faster (3.8 to 8.7), driven by the increasing weight of large consortia.
- Solo authored papers have declined from 14.6% to 2.5%, and the share of papers with 10+ authors has jumped from 3.1% to 30.3%.
- The composition shift shows a genuine reorganisation of research: small team papers (2–5 authors) are being replaced by mid size (6–10) and large (11–20) team papers, not simply a few mega papers pulling the average.

**Part B – Prolific Author Network**
- The scoped network (post‑2014, ≤50 authors, authors with ≥100 papers) resolves into **236 distinct collaboration communities**.
- Leading communities correspond to major research areas: heart failure trials (McMurray, Solomon, Zannad), Alzheimer’s biomarkers (Zetterberg, Blennow, Jack), and oncology consortia.
- The strongest pairwise collaboration (Blennow–Zetterberg, 562 joint papers) represents a well documented, productive scientific partnership, validating that the high threshold network captures real collaborations.

### Caveats

- **Author names are not disambiguated.** A name like "J Smith" is treated as a single node, over merging distinct researchers and splitting authors who use different name variants. At the high prolific threshold (≥100 papers), this effect is limited for the top results, which are dominated by uniquely named leaders, but it remains a structural limitation of any name based network.

- **Large consortia papers are excluded by design.** Papers with more than 50 authors are dropped from the pairwise graph because their `k choose 2` pair count would dominate the edge set and exhaust memory. This excludes only 0.14% of papers but avoids billions of spurious pairs. The network therefore describes normal scale collaboration, not mega consortium science.

- **The network is restricted to post‑2014 and to prolific authors.** Affiliation coverage is unreliable before 2014 (see EDA), and one off authors add noise without structure. The network is thus a view of *recurring, recent* collaboration, not all authorship. Part A's metrics, which use the full corpus, provide the unrestricted perspective.

- **Thresholds are judgement calls.** `MAX_AUTHORS`, `MIN_PAPERS`, and `MIN_COLLAB` shape the network. They were chosen to produce a readable, meaningful graph and are explicit at the top of Section 4. Different values would yield a denser or sparser network. The metrics in Part A do not depend on any of them.

### Implications

These patterns reflect structural changes in biomedical research: increasing specialisation, multi centre clinical trials, and the emergence of "invisible colleges" of highly collaborative researchers. The network approach reveals social organisation that complements the temporal team size trends, showing how distinct research communities form and persist. This dual view, macro trends plus micro community structure, provides a richer picture of the modern scientific workforce.

**Next up:** proceed to `06_keyword_trends.ipynb` for author‑keyword analysis and topic emergence. Start the timeline around 2015 (keywords are near‑empty before then, per EDA section 7a) and normalise by articles per year so that rising keywords reflect real adoption rather than corpus growth.